In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
humansintheloop_semantic_segmentation_of_aerial_imagery_path = kagglehub.dataset_download('humansintheloop/semantic-segmentation-of-aerial-imagery')

print('Data source import complete.')
print(humansintheloop_semantic_segmentation_of_aerial_imagery_path)

Using Colab cache for faster access to the 'semantic-segmentation-of-aerial-imagery' dataset.
Data source import complete.
/kaggle/input/semantic-segmentation-of-aerial-imagery


# Лабораторная работа 1 (CV) — Семантическая сегментация

**Выбранный датасет:** Semantic Segmentation of Aerial Imagery (Dubai Dataset)
**Задача:** Семантическая сегментация спутниковых снимков для картографии и урбанистики.

Этот ноутбук содержит:
1. Подключение и визуализацию датасета.
2. Создание бейзлайна с использованием `segmentation_models.pytorch`.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Подготовка данных

Определим пути к данным и маппинг цветов.

In [ ]:
# Проверяем возможные пути к датасету в Kaggle и Colab
possible_paths = [
    "/kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery",
    "/kaggle/input/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset",
    "/kaggle/input/semantic-segmentation-of-aerial-imagery"
]

# Если мы в Google Colab и датасет скачан через kagglehub
if 'humansintheloop_semantic_segmentation_of_aerial_imagery_path' in globals() or 'humansintheloop_semantic_segmentation_of_aerial_imagery_path' in locals():
    base_path = humansintheloop_semantic_segmentation_of_aerial_imagery_path
    possible_paths.insert(0, os.path.join(base_path, "Semantic segmentation dataset"))
    possible_paths.insert(0, base_path)
else:
    try:
        import kagglehub
        colab_path = kagglehub.dataset_download('humansintheloop/semantic-segmentation-of-aerial-imagery')
        possible_paths.insert(0, os.path.join(colab_path, "Semantic segmentation dataset"))
        possible_paths.insert(0, colab_path)
    except:
        pass

DATA_DIR = None
for p in possible_paths:
    if os.path.exists(p):
        if os.path.exists(os.path.join(p, "Tile 1")):
            DATA_DIR = p
            break

if DATA_DIR is None:
    print("ВНИМАНИЕ: Папка с датасетом не найдена!")
else:
    print(f"Используем директорию: {DATA_DIR}")

# Цвета классов из classes.json (в формате RGB)
classes_info = [
    {'name': 'Building', 'color': [60, 16, 152]},      # #3C1098
    {'name': 'Land', 'color': [132, 41, 246]},         # #8429F6
    {'name': 'Road', 'color': [110, 193, 228]},        # #6EC1E4
    {'name': 'Vegetation', 'color': [254, 221, 58]},   # #FEDD3A
    {'name': 'Water', 'color': [226, 169, 41]},        # #E2A929
    {'name': 'Unlabeled', 'color': [155, 155, 155]}    # #9B9B9B
]

COLORS = np.array([c['color'] for c in classes_info])
CLASS_NAMES = [c['name'] for c in classes_info]
NUM_CLASSES = len(classes_info)

def rgb_to_mask(img, colors):
    """Преобразует RGB-маску в 2D маску с индексами классов."""
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    for idx, color in enumerate(colors):
        diff = np.abs(img - color).sum(axis=-1)
        mask[diff < 10] = idx
    return mask

Используем директорию: /kaggle/input/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset


In [ ]:
# Соберем пути ко всем изображениям и маскам
image_paths = []
mask_paths = []

if DATA_DIR is not None:
    for tile_idx in range(1, 9):  # Tile 1 до Tile 8
        tile_dir = os.path.join(DATA_DIR, f"Tile {tile_idx}")
        images_dir = os.path.join(tile_dir, "images")
        masks_dir = os.path.join(tile_dir, "masks")

        if os.path.exists(images_dir) and os.path.exists(masks_dir):
            # В датасете иногда могут быть не только jpg/png
            imgs = sorted(glob.glob(os.path.join(images_dir, "*.jpg")) + glob.glob(os.path.join(images_dir, "*.png")))
            msks = sorted(glob.glob(os.path.join(masks_dir, "*.png")) + glob.glob(os.path.join(masks_dir, "*.jpg")))

            # Убедимся, что пары совпадают по имени (без учета расширения)
            for img_path in imgs:
                base_name = os.path.splitext(os.path.basename(img_path))[0]
                msk_path_png = os.path.join(masks_dir, base_name + ".png")
                msk_path_jpg = os.path.join(masks_dir, base_name + ".jpg")

                if os.path.exists(msk_path_png):
                    image_paths.append(img_path)
                    mask_paths.append(msk_path_png)
                elif os.path.exists(msk_path_jpg):
                    image_paths.append(img_path)
                    mask_paths.append(msk_path_jpg)

print(f"Найдено {len(image_paths)} изображений и {len(mask_paths)} масок.")

if len(image_paths) == 0:
    raise ValueError("Датасет не найден! Проверьте путь DATA_DIR (в предыдущей ячейке).")

# Разделение на train и val
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=42
)
print(f"Train size: {len(train_imgs)}, Validation size: {len(val_imgs)}")

Найдено 72 изображений и 72 масок.
Train size: 57, Validation size: 15


In [ ]:
class DubaiDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Читаем изображение
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Читаем RGB маску
        mask_rgb = cv2.imread(self.mask_paths[idx])
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        # Преобразуем RGB маску в индексы классов
        mask = rgb_to_mask(mask_rgb, COLORS)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']

        return img, mask.long()

In [ ]:
# Изображения в датасете большого размера, поэтому для бейзлайна мы их ресайзим до 512x512.
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_dataset = DubaiDataset(train_imgs, train_masks, transform=train_transform)
val_dataset = DubaiDataset(val_imgs, val_masks, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

## 2. Создание бейзлайна (Baseline)
Используем `segmentation_models.pytorch` для загрузки архитектуры Unet с энкодером ResNet34.

In [ ]:
import json

# Используем CrossEntropy для многоклассовой сегментации
criterion = nn.CrossEntropyLoss()

# Выбранные метрики для логирования (будем рассчитывать IoU для каждого класса и усреднять)
def calculate_iou(pred_mask, true_mask, num_classes):
    ious = []
    pred_mask = torch.argmax(pred_mask, dim=1)
    for cls in range(num_classes):
        pred_inds = pred_mask == cls
        target_inds = true_mask == cls
        intersection = (pred_inds[target_inds]).long().sum().item()
        union = pred_inds.long().sum().item() + target_inds.long().sum().item() - intersection
        if union == 0:
            ious.append(float('nan'))  # Если класс не присутствует ни там, ни там
        else:
            ious.append(intersection / max(union, 1))
    return np.nanmean(ious) # mIoU

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, model_name="model"):
    """
    Универсальная функция для обучения, валидации, локального логирования (в json)
    и сохранения лучших весов модели.
    """
    history = {'train_loss': [], 'val_loss': [], 'val_mIoU': []}
    best_iou = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Валидация
        model.eval()
        val_loss = 0.0
        val_iou = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                
                loss = criterion(outputs, masks)
                val_loss += loss.item() * images.size(0)
                
                # Рассчет mIoU
                val_iou += calculate_iou(outputs, masks, NUM_CLASSES) * images.size(0)
                
        val_loss /= len(val_loader.dataset)
        val_iou /= len(val_loader.dataset)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mIoU'].append(val_iou)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_iou:.4f}")
        
        # Сохранение весов лучшей модели (пункт 7)
        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), f'{model_name}_best.pth')
            print(f"--> Сохранены улучшенные веса: {best_iou:.4f}")
            
    # Локальное логирование: сохраняем историю обучения (пункт 6)
    with open(f'{model_name}_history.json', 'w') as f:
        json.dump(history, f)
        
    print(f"Обучение {model_name} завершено. Логи локально сохранены.")
    return history

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
print("Инициализация сверточной архитектуры: Unet + ResNet34")
model_cnn = smp.Unet(
    encoder_name="resnet34",        # сверточный энкодер
    encoder_weights="imagenet",     # предобученные веса на ImageNet
    in_channels=3,                  # RGB изображения
    classes=NUM_CLASSES             # количество классов
).to(device)

optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-4)

# Запуск обучения сверточного бейзлайна (пункт 6)
NUM_EPOCHS = 10  # Увеличим число эпох до 10 для более репрезентативных результатов
history_cnn = train_model(
    model_cnn, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer_cnn, 
    num_epochs=NUM_EPOCHS, 
    model_name="baseline_cnn_resnet34"
)

Epoch [1/5] | Train Loss: 1.7423 | Val Loss: 1.4896 | Val mIoU: 0.1690
Epoch [2/5] | Train Loss: 1.3717 | Val Loss: 1.4484 | Val mIoU: 0.1852
Epoch [3/5] | Train Loss: 1.2197 | Val Loss: 1.2491 | Val mIoU: 0.2161
Epoch [4/5] | Train Loss: 1.1239 | Val Loss: 1.1704 | Val mIoU: 0.2457
Epoch [5/5] | Train Loss: 1.0868 | Val Loss: 1.1206 | Val mIoU: 0.2818
Training Pipeline (Baseline) Complete!


## 3. Создание трансформерного бейзлайна
Используем тот же `smp.Unet`, но в качестве энкодера — **MiT-B0** (Mix Vision Transformer).

In [ ]:
print("Инициализация трансформерной архитектуры: Unet + MiT (Mix Vision Transformer)")
# mit_b0 - легкий энкодер из серии SegFormer
model_transformer = smp.Unet(
    encoder_name="mit_b0",          # трансформерный энкодер
    encoder_weights="imagenet",     # предобученные веса на ImageNet
    in_channels=3,                  # RGB изображения
    classes=NUM_CLASSES             # количество классов
).to(device)

optimizer_transformer = torch.optim.Adam(model_transformer.parameters(), lr=1e-4)

# Запуск обучения трансформерного бейзлайна (пункт 6)
history_transformer = train_model(
    model_transformer, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer_transformer, 
    num_epochs=NUM_EPOCHS, 
    model_name="baseline_transformer_mit_b0"
)

In [ ]:
# Пункт 7: Оценка качества бейзлайн-моделей (Сравнение результатов)

plt.figure(figsize=(15, 5))

# График зависимости Val Loss (Функция потерь на валидации)
plt.subplot(1, 2, 1)
plt.plot(history_cnn['val_loss'], label='CNN (ResNet34) Val Loss', marker='o', color='blue')
plt.plot(history_transformer['val_loss'], label='Transformer (MiT-B0) Val Loss', marker='s', color='orange')
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss Value (CrossEntropy)')
plt.legend()
plt.grid(True)

# График зависимости Val mIoU (Метрика на валидации)
plt.subplot(1, 2, 2)
plt.plot(history_cnn['val_mIoU'], label='CNN (ResNet34) Val mIoU', marker='o', color='blue')
plt.plot(history_transformer['val_mIoU'], label='Transformer (MiT-B0) Val mIoU', marker='s', color='orange')
plt.title('Validation mIoU Comparison')
plt.xlabel('Epoch')
plt.ylabel('mIoU Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Печать итогов
best_cnn_iou = max(history_cnn['val_mIoU'])
best_tf_iou = max(history_transformer['val_mIoU'])

print(f"Лучший mIoU у CNN на валидации: {best_cnn_iou:.4f}")
print(f"Лучший mIoU у Transformer на валидации: {best_tf_iou:.4f}")

if best_cnn_iou > best_tf_iou:
    print("-> Сверточный бейзлайн показал себя лучше.")
else:
    print("-> Трансформерный бейзлайн показал себя лучше.")
print("\n* Веса лучших моделей (baseline_cnn_resnet34_best.pth и baseline_transformer_mit_b0_best.pth) сохранены в текущей рабочей директории.")